# p53 Mutant MD Simulation with ESMFold API

**IMPORTANT: This notebook requires 2 steps:**
1. Run Cell 1 to install conda (runtime will restart automatically)
2. After restart, run Cell 2 onward

Uses ESMFold API for structure prediction and OpenMM for MD.

In [ ]:
# @title Step 1: Install Conda (RUNTIME WILL RESTART)
# Run this cell FIRST. The runtime will restart automatically.
# After restart, skip this cell and run Step 2.

import os
if not os.path.exists('/usr/local/conda-meta'):
    print("Installing condacolab...")
    !pip install -q condacolab
    import condacolab
    condacolab.install()  # This restarts the runtime
else:
    print("Conda already installed. Proceed to Step 2.")

In [ ]:
# @title Step 2: Install Packages (Run after restart)
import condacolab
condacolab.check()

print("Installing packages via conda...")
!conda install -y -c conda-forge openmm pdbfixer mdtraj -q
!pip install -q requests matplotlib

print("\nDone! Proceed to next cell.")

In [ ]:
# @title Step 3: Imports
import requests
import time
import os
import sys
import numpy as np
import matplotlib.pyplot as plt

from openmm import *
from openmm.app import *
from openmm.unit import *
from pdbfixer import PDBFixer
import mdtraj as md

os.makedirs("structures", exist_ok=True)
os.makedirs("trajectories", exist_ok=True)

print("All imports successful!")

In [ ]:
# @title Step 4: ESMFold API Function

def predict_structure(sequence, max_retries=3):
    """Predict structure using ESMFold API."""
    url = "https://api.esmatlas.com/foldSequence/v1/pdb/"
    for attempt in range(max_retries):
        try:
            print(f"  Calling ESMFold API (attempt {attempt+1})...")
            r = requests.post(url, data=sequence, 
                            headers={'Content-Type': 'text/plain'},
                            timeout=300)
            if r.status_code == 200:
                print("  Success!")
                return r.text
            print(f"  Error {r.status_code}, waiting 30s...")
            time.sleep(30)
        except Exception as e:
            print(f"  {e}")
            time.sleep(10)
    raise RuntimeError("ESMFold API failed after retries")

print("ESMFold function ready.")

In [ ]:
# @title Step 5: p53 Sequence

P53_FULL = (
    "MEEPQSDPSVEPPLSQETFSDLWKLLPENNVLSPLPSQAMDDLMLSPDDIEQWFTEDPGP"
    "DEAPRMPEAAPPVAPAPAAPTPAAPAPAPSWPLSSSVPSQKTYQGSYGFRLGFLHSGTAK"
    "SVTCTYSPALNKMFCQLAKTCPVQLWVDSTPPPGTRVRAMAIYKQSQHMTEVVRRCPHHE"
    "RCSDSDGLAPPQHLIRVEGNLRVEYLDDRNTFRHSVVVPYEPPEVGSDCTTIHYNYMCNS"
    "SCMGGMNRRPILTIITLEDSSGNLLGRNSFEVRVCACPGRDRRTEEENLRKKGEPHHELP"
    "PGSTKRALPNNTSSSPQPKKKPLDGEYFTLQIRGRERFEMFRELNEALELKDAQAGKEPG"
    "GSRAHSSHLKSKKGQSTSRHKKLMFKTEGPDSD"
)

CORE_START, CORE_END = 94, 312
P53_CORE = P53_FULL[CORE_START-1:CORE_END]

print(f"p53 core domain: {len(P53_CORE)} residues (positions {CORE_START}-{CORE_END})")

In [ ]:
# @title Step 6: Configuration - EDIT THIS CELL

# ============ EDIT THESE VALUES ============
TARGET_MUTATION = "R175H"        # Destabilizing cancer mutation
RESCUE_MUTATIONS = ["N239Y"]     # Putative rescue mutation(s)
PRODUCTION_NS = 1.0              # Production MD length (nanoseconds)
# ===========================================

ALL_MUTATIONS = [TARGET_MUTATION] + RESCUE_MUTATIONS
MUTANT_NAME = "_".join(ALL_MUTATIONS)

# MD parameters
TIMESTEP_FS = 2.0
EQUILIBRATION_STEPS = 50000      # 100 ps
PRODUCTION_STEPS = int(PRODUCTION_NS * 1e6 / TIMESTEP_FS)

print(f"Configuration:")
print(f"  Target: {TARGET_MUTATION}")
print(f"  Rescue: {RESCUE_MUTATIONS}")
print(f"  Production: {PRODUCTION_NS} ns ({PRODUCTION_STEPS} steps)")

In [ ]:
# @title Step 7: Generate Mutant Sequence & Predict Structure

# Apply mutations to sequence
mutant_seq = P53_CORE
print("Applying mutations:")
for mut in ALL_MUTATIONS:
    wt_aa = mut[0]
    position = int(mut[1:-1])
    mut_aa = mut[-1]
    
    # Convert to core domain index
    core_idx = position - CORE_START
    
    if core_idx < 0 or core_idx >= len(mutant_seq):
        raise ValueError(f"Position {position} outside core domain")
    if mutant_seq[core_idx] != wt_aa:
        raise ValueError(f"Expected {wt_aa} at {position}, found {mutant_seq[core_idx]}")
    
    mutant_seq = mutant_seq[:core_idx] + mut_aa + mutant_seq[core_idx+1:]
    print(f"  {mut}: position {position} (index {core_idx})")

# Predict structure
print(f"\nPredicting structure for {MUTANT_NAME}...")
pdb_string = predict_structure(mutant_seq)

raw_path = f"structures/{MUTANT_NAME}_raw.pdb"
with open(raw_path, 'w') as f:
    f.write(pdb_string)
print(f"Saved: {raw_path}")

In [ ]:
# @title Step 8: Fix Structure with PDBFixer

print("Fixing structure with PDBFixer...")
fixer = PDBFixer(filename=raw_path)

# Find and fix missing atoms (including terminal OXT)
fixer.findMissingResidues()
fixer.findMissingAtoms()
fixer.addMissingAtoms()
print(f"  Added missing atoms")

# Add hydrogens at pH 7
fixer.addMissingHydrogens(7.0)
print(f"  Added hydrogens")

# Save fixed structure
fixed_path = f"structures/{MUTANT_NAME}_fixed.pdb"
with open(fixed_path, 'w') as f:
    PDBFile.writeFile(fixer.topology, fixer.positions, f)
print(f"  Saved: {fixed_path}")

print(f"\nFixed structure: {fixer.topology.getNumAtoms()} atoms")

In [ ]:
# @title Step 9: Solvate & Create System

print("Creating simulation system...")

# Force field
forcefield = ForceField('amber14-all.xml', 'amber14/tip3pfb.xml')

# Create modeller from fixed structure
modeller = Modeller(fixer.topology, fixer.positions)

# Add solvent
print("  Adding solvent (1 nm padding)...")
modeller.addSolvent(
    forcefield,
    model='tip3p',
    padding=1.0*nanometer,
    ionicStrength=0.15*molar
)

# Save solvated structure
solvated_path = f"structures/{MUTANT_NAME}_solvated.pdb"
with open(solvated_path, 'w') as f:
    PDBFile.writeFile(modeller.topology, modeller.positions, f)
print(f"  Saved: {solvated_path}")

# Create system
print("  Creating OpenMM system...")
system = forcefield.createSystem(
    modeller.topology,
    nonbondedMethod=PME,
    nonbondedCutoff=1.0*nanometer,
    constraints=HBonds
)

print(f"\nSystem ready: {system.getNumParticles()} atoms")

In [ ]:
# @title Step 10: Energy Minimization

print("Setting up simulation...")
integrator = LangevinMiddleIntegrator(300*kelvin, 1/picosecond, TIMESTEP_FS*femtoseconds)
simulation = Simulation(modeller.topology, system, integrator)
simulation.context.setPositions(modeller.positions)

print("\nRunning energy minimization...")
e_initial = simulation.context.getState(getEnergy=True).getPotentialEnergy()
print(f"  Initial: {e_initial}")

simulation.minimizeEnergy(maxIterations=1000)

e_final = simulation.context.getState(getEnergy=True).getPotentialEnergy()
print(f"  Final: {e_final}")

# Get minimized positions
positions = simulation.context.getState(getPositions=True).getPositions()

# Save minimized structure
min_path = f"structures/{MUTANT_NAME}_minimized.pdb"
with open(min_path, 'w') as f:
    PDBFile.writeFile(simulation.topology, positions, f)
print(f"  Saved: {min_path}")

In [ ]:
# @title Step 11: NPT Equilibration

print("Setting up NPT equilibration...")

# Add barostat for pressure control
system.addForce(MonteCarloBarostat(1*bar, 300*kelvin))

# New integrator
integrator = LangevinMiddleIntegrator(300*kelvin, 1/picosecond, TIMESTEP_FS*femtoseconds)

# New simulation with barostat
simulation = Simulation(modeller.topology, system, integrator)
simulation.context.setPositions(positions)
simulation.context.setVelocitiesToTemperature(300*kelvin)

# Run equilibration
print(f"Running NPT equilibration (100 ps)...")
simulation.step(EQUILIBRATION_STEPS)

# Save equilibrated structure
eq_path = f"structures/{MUTANT_NAME}_equilibrated.pdb"
eq_positions = simulation.context.getState(getPositions=True).getPositions()
with open(eq_path, 'w') as f:
    PDBFile.writeFile(simulation.topology, eq_positions, f)
print(f"  Saved: {eq_path}")

In [ ]:
# @title Step 12: Production MD

print(f"Starting production MD ({PRODUCTION_NS} ns)...")
print("Progress will be shown below.\n")

# Output files
traj_path = f"trajectories/{MUTANT_NAME}.dcd"
log_path = f"trajectories/{MUTANT_NAME}.csv"

# Add reporters
simulation.reporters.append(DCDReporter(traj_path, 5000))  # Every 10 ps
simulation.reporters.append(StateDataReporter(
    log_path, 5000,
    step=True, time=True, potentialEnergy=True, temperature=True,
    progress=True, remainingTime=True, speed=True,
    totalSteps=PRODUCTION_STEPS
))
simulation.reporters.append(StateDataReporter(
    sys.stdout, 50000,  # Print every 100 ps
    step=True, time=True, potentialEnergy=True, temperature=True,
    progress=True, remainingTime=True, speed=True,
    totalSteps=PRODUCTION_STEPS
))

# Run production
simulation.step(PRODUCTION_STEPS)

print(f"\nProduction complete!")
print(f"  Trajectory: {traj_path}")
print(f"  Log: {log_path}")

In [ ]:
# @title Step 13: RMSD Analysis

print("Analyzing trajectory...")

# Load trajectory
traj = md.load(traj_path, top=eq_path)
print(f"  Loaded {traj.n_frames} frames")

# Select protein atoms
protein_idx = traj.topology.select('protein')
protein = traj.atom_slice(protein_idx)
print(f"  Protein: {protein.n_atoms} atoms")

# Calculate RMSD vs first frame
rmsd = md.rmsd(protein, protein, 0) * 10  # Convert nm to Angstrom
time_ns = np.arange(len(rmsd)) * TIMESTEP_FS * 5000 / 1e6

# Plot
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(time_ns, rmsd, 'b-', linewidth=0.5, alpha=0.8)
ax.axhline(2.0, color='green', linestyle='--', label='Stable (<2 Å)')
ax.axhline(3.0, color='orange', linestyle='--', label='Moderate (<3 Å)')
ax.axhline(5.0, color='red', linestyle='--', label='Unstable (>5 Å)')

ax.set_xlabel('Time (ns)', fontsize=12)
ax.set_ylabel('RMSD (Å)', fontsize=12)
ax.set_title(f'{MUTANT_NAME} - Backbone RMSD', fontsize=14)
ax.legend(loc='upper left')
ax.grid(True, alpha=0.3)
ax.set_ylim(0, max(8, rmsd.max() * 1.1))

plt.tight_layout()
fig_path = f"trajectories/{MUTANT_NAME}_rmsd.png"
plt.savefig(fig_path, dpi=150)
plt.show()

# Statistics
print(f"\n{'='*50}")
print(f"RMSD Statistics for {MUTANT_NAME}")
print(f"{'='*50}")
print(f"  Mean RMSD:  {np.mean(rmsd):.2f} Å")
print(f"  Final RMSD: {rmsd[-1]:.2f} Å")
print(f"  Max RMSD:   {np.max(rmsd):.2f} Å")
print(f"  Std RMSD:   {np.std(rmsd):.2f} Å")

# Verdict
final_rmsd = rmsd[-1]
if final_rmsd < 2.5:
    print(f"\n✅ STABLE: Structure maintains fold (RMSD = {final_rmsd:.2f} Å)")
elif final_rmsd < 4.0:
    print(f"\n⚠️ MODERATE: Some conformational changes (RMSD = {final_rmsd:.2f} Å)")
else:
    print(f"\n❌ UNSTABLE: Significant unfolding (RMSD = {final_rmsd:.2f} Å)")

In [ ]:
# @title Step 14: Summary

print("="*60)
print(f"SIMULATION COMPLETE: {MUTANT_NAME}")
print("="*60)

print(f"\nMutations:")
print(f"  Target (destabilizing): {TARGET_MUTATION}")
print(f"  Rescue mutations: {RESCUE_MUTATIONS}")

print(f"\nSimulation parameters:")
print(f"  Equilibration: 100 ps NPT")
print(f"  Production: {PRODUCTION_NS} ns")
print(f"  Temperature: 300 K")
print(f"  Pressure: 1 bar")

print(f"\nOutput files:")
print(f"  structures/{MUTANT_NAME}_raw.pdb          (ESMFold prediction)")
print(f"  structures/{MUTANT_NAME}_fixed.pdb        (PDBFixer output)")
print(f"  structures/{MUTANT_NAME}_solvated.pdb     (Solvated system)")
print(f"  structures/{MUTANT_NAME}_equilibrated.pdb (After NPT)")
print(f"  trajectories/{MUTANT_NAME}.dcd            (MD trajectory)")
print(f"  trajectories/{MUTANT_NAME}.csv            (Energy log)")
print(f"  trajectories/{MUTANT_NAME}_rmsd.png       (RMSD plot)")

print(f"\nNext steps:")
print(f"  1. Compare with wild-type p53 simulation")
print(f"  2. Compare with {TARGET_MUTATION}-only (no rescue)")
print(f"  3. Run longer simulations (10-50 ns)")
print(f"  4. Try other rescue candidates from Pareto front")